In [ ]:
import numpy as np
import cupy as cp
from scipy.stats import norm
import time
import h5py
from scipy.linalg import solve_banded
from concurrent.futures import ProcessPoolExecutor, as_completed
# import os

In [5]:
# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)

# r, lamb, v_bar, epsilon, rho, Y0, T = eta
Hes_eta = (0.03, 2, 0.05, 0.5, -0.7, 0.05, 1.5)

In [2]:
# step1. generate parameter combinations
#1) BS
def generate_BS_params(n_sets, seed=None):
    if seed is not None:
        np.random.seed(seed)

    S0 = np.random.normal(loc=1.0, scale=0.2, size=n_sets)  # S0 ~ N(1, 0.2^2) Greek 계산시 delta를 구할 경우 1로 안둠.
    r = np.random.uniform(0, 0.1, n_sets)      # r ~ U(0, 0.1)
    sigma = np.random.uniform(0.001, 1, n_sets)# σ ~ U(0.001, 1)
    T = np.random.uniform(0.1, 3, n_sets)      # T ~ U(0.1, 3)
    K = 1                                      # K = 1

    params = np.stack([S0, K, r, sigma, T], axis=1)
    return params

#2) heston
def generate_heston_params(n_sets, seed=None):
    if seed is not None:
        np.random.seed(seed)

    r = np.random.uniform(0, 0.1, n_sets)      # r ~ U(0, 0.1)
    lamb = np.random.beta(2, 18, n_sets) * 20  # λ ~ Beta(2, 18) × 20
    v_bar = np.random.beta(1, 19, n_sets)      # v_bar ~ Beta(1, 19)
    epsilon = np.random.uniform(0.1, 1, n_sets)# ξ ~ U(0.1, 1)
    rho = np.random.uniform(-1, 0, n_sets)     # ρ ~ U(-1, 0)
    Y0 = np.random.beta(1, 19, n_sets)         # Y₀ ~ Beta(1, 19)
    T = np.random.uniform(0.1, 3, n_sets)      # T ~ U(0.1, 3)
        
    params = np.stack([r, lamb, v_bar, epsilon, rho, Y0, T], axis=1)
    return params

def filter_milestein(params): # milestein condition
    _, lamb, v_bar, epsilon, *_ = params.T
    mask = 4 * lamb * v_bar > epsilon**2
    return params[mask]

def generate_valid_params(n_sets, seed=None):
    raw = generate_heston_params(int(n_sets * 1.1), seed)
    filtered = filter_milestein(raw)
    
    while len(filtered) < n_sets:
        extra = generate_heston_params(n_sets)
        extra = filter_milestein(extra)
        filtered = np.vstack([filtered, extra])
    
    return filtered[:n_sets]

def min_max_normalize(data):
    return (data - data.min()) / (data.max() - data.min())

In [ ]:
#step1-1
BS_paras = generate_BS_params(n_sets=100*(2**16), seed=1234)
BS_paras_scaled = BS_paras.copy()

for col in range(5):
    if col == 0:
        BS_paras_scaled[:, col] = (BS_paras[:, col] - 1.0) / 0.2
    else:
        BS_paras_scaled[:, col] = min_max_normalize(BS_paras[:, col])

In [8]:
#step1-2 
Hes_paras = generate_valid_params(n_sets=100*(2**16), seed=1234) # 4.1초

Hes_paras_scaled = Hes_paras.copy()
for col in range(7):
    Hes_paras_scaled[:, col] = min_max_normalize(Hes_paras[:, col])

ETA_PATH = "/mnt/d/heston_eta.h5" # 10초
with h5py.File(ETA_PATH, "w") as f:
    f.create_dataset("etas", data=Hes_paras,
                     maxshape=(None, 7), chunks=(10240, 7), compression="gzip")

"""
print(f"\n파라미터 범위 확인:")
names = ['r', 'λ', 'v_bar', 'ξ', 'ρ', 'Y₀', 'T']
for i, name in enumerate(names):
    print(f"{name}: min={params[:,i].min():.4f}, max={params[:,i].max():.4f}, mean={params[:,i].mean():.4f}")
"""

'\nprint(f"\n파라미터 범위 확인:")\nnames = [\'r\', \'λ\', \'v_bar\', \'ξ\', \'ρ\', \'Y₀\', \'T\']\nfor i, name in enumerate(names):\n    print(f"{name}: min={params[:,i].min():.4f}, max={params[:,i].max():.4f}, mean={params[:,i].mean():.4f}")\n'

In [3]:
# step2. generate path
# 1) Heston
def generate_heston_paths(eta, n_paths=2**10, dt=0.001):
    r, lamb, v_bar, epsilon, rho, Y0, T = eta
    n_steps = int(T / dt)

    W1 = np.random.randn(n_paths, n_steps)
    W2 = np.random.randn(n_paths, n_steps)
    dWx = np.sqrt(dt) * W1
    dWy = np.sqrt(dt) * (rho * W1 + np.sqrt(1 - rho**2) * W2)

    X = np.zeros((n_paths, n_steps + 1))
    Y = np.zeros((n_paths, n_steps + 1))
    X[:, 0] = 0.0
    Y[:, 0] = Y0

    for i in range(n_steps):
        Y_t = np.maximum(Y[:, i], 0)
        X[:, i+1] = X[:, i] + (r - 0.5 * Y_t) * dt + np.sqrt(Y_t) * dWx[:, i]
        Y[:, i+1] = (Y_t
                     + lamb * (v_bar - Y_t) * dt
                     + epsilon * np.sqrt(Y_t) * dWy[:, i]
                     + 0.25 * epsilon**2 * (dWy[:, i]**2 - dt))

    XT = X[:, -1]
    YT = Y[:, -1]
    MT = X.min(axis=1)
    mask = filter_paths(XT, T)
    return XT, YT, MT, mask

# 연율화 수익률 평균/분산 필터링 (배치 전체 기준)
def filter_paths(XT, T):
    annual_return = XT / T
    check_mean = np.abs(annual_return.mean()) <= 0.3
    check_var = annual_return.var() <= 1.0
    return bool(check_mean and check_var)

In [ ]:
#step2-2
ETA_PATH  = "/mnt/d/heston_eta.h5"
SAVE_PATH = "/mnt/d/heston_dataset.h5"


for i in range(9,10):
    print(i)
    with h5py.File(ETA_PATH, "r") as ef:
        Hes_paras = ef["etas"][:]  # (2**16)*100개, 여기서 target*num개 만큼씩 가져오게 해도 됨.

    target = (2**16) * 10 # 3.6h 걸림
    true = 0
    fail = 0 # 1116 + 1128 + 1114 + 1111 + 1084 + 1127 + 1143 + 1058 + 1089 + 1054
    num = i # 9까지, done : 0, 1, 2, 3, 4, 5, 6, 7, 8, 9 
    start = time.time()

    # CPU 병렬, 멀티 쓰레딩은 성능 안좋음
    N_WORKERS = 14 # 1만개 기준 = 14:207s/28:192s, 이용률 2배차이
    BATCH_SIZE = N_WORKERS * 10


    def _worker(args):
        eta, n_paths, dt = args
        return generate_heston_paths(eta, n_paths=n_paths, dt=dt)

    with h5py.File(SAVE_PATH, "a") as f, ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        if "paths" not in f:
            f.create_dataset("paths", shape=(0, 4), maxshape=(None, 4),
                            dtype="float64", chunks=(10240, 4), compression="gzip")

        for i in range(0, target, BATCH_SIZE):
            eta_batch = Hes_paras[i + target*num : i + BATCH_SIZE + target*num]
                    
            futures = {executor.submit(_worker, (eta, 2**10, 0.001)): (i + target*num + k, eta)
                    for k, eta in enumerate(eta_batch)}

            for future in as_completed(futures):
                ori_idx, eta = futures[future]
                XT, YT, MT, mask = future.result()

                if mask:
                    rows = np.column_stack([
                        np.full(len(XT), ori_idx),
                        XT, YT, MT
                    ])
                    f["paths"].resize(f["paths"].shape[0] + len(rows), axis=0)
                    f["paths"][-len(rows):] = rows

                    true += 1
                    if true % 20000 == 0:
                        print(f"[{true}/{target}] fail: {fail}")
                else:
                    fail += 1

                if target <= true:
                    break
            else:
                continue
            break
        else:
            Hes_paras = generate_valid_params(n_sets=(target - true), seed=None)

    elapsed = time.time() - start
    print(f"done. true: {true}, fail: {fail}")
    print(f"elapsed: {elapsed:.1f}s ({elapsed/3600:.2f}h)")

9
[20000/655360] fail: 34
[40000/655360] fail: 69
[60000/655360] fail: 105
[80000/655360] fail: 126
[100000/655360] fail: 163
[120000/655360] fail: 194
[140000/655360] fail: 234
[160000/655360] fail: 259
[180000/655360] fail: 291
[200000/655360] fail: 322
[220000/655360] fail: 354
[240000/655360] fail: 382
[260000/655360] fail: 408
[280000/655360] fail: 434
[300000/655360] fail: 460
[320000/655360] fail: 483
[340000/655360] fail: 513
[360000/655360] fail: 538
[380000/655360] fail: 565
[400000/655360] fail: 599
[420000/655360] fail: 627
[440000/655360] fail: 666
[460000/655360] fail: 701
[480000/655360] fail: 743
[500000/655360] fail: 782
[520000/655360] fail: 828
[540000/655360] fail: 865
[560000/655360] fail: 895
[580000/655360] fail: 930
[600000/655360] fail: 963
[620000/655360] fail: 1000
[640000/655360] fail: 1036
done. true: 654306, fail: 1054
elapsed: 15000.2s (4.17h)


In [ ]:
# 1. closed-form
# 1)vanilla
def BS_vanilla(eta, type='call'):
    # type : 'call' or 'put'
    S0, K, r, sigma, T = eta

    sqT  = sigma * np.sqrt(T)
    disc = np.exp(-r * T)

    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / sqT
    d2 = d1 - sqT

    if type == 'call':
        price = S0 * norm.cdf(d1) - K * disc * norm.cdf(d2)
    elif type == 'put':
        price = K * disc * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    return price



#2) down-and-out call
def BS_barrier(eta, type='call'):
    # type : 'call' or 'put'

    S0, K, r, sigma, T  = eta
    B = 0.8
    
    lam = r / sigma**2 + 0.5          
    sqT = sigma * np.sqrt(T)          
    disc = np.exp(-r * T)                     

    # vanilla
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / sqT
    d2 = d1 - sqT

    def vanilla_call():
        return S0 * norm.cdf(d1) - K * disc * norm.cdf(d2)

    def vanilla_put():
        return K * disc * norm.cdf(-d2) - S0 * norm.cdf(-d1)


    coef1 = S0 * (B / S0)**(2 * lam)           
    coef2 = K * disc * (B / S0)**(2 * lam - 2) 
    x1 = np.log(B**2 / (S0 * K)) / sqT + lam * sqT
    x2 = x1 - sqT

    if type == 'call':
        if B > K:
            raise ValueError("change B to be smaller than K")
        
        C_do = (vanilla_call() 
                - coef1 * norm.cdf(x1) 
                + coef2 * norm.cdf(x2))
        return C_do
    elif type == 'put':
        if B > K:
            raise ValueError("change B to be smaller than K")

        x_h1 = np.log(S0 / B) / sqT + lam * sqT
        x_h2 = x_h1 - sqT

        y1 = np.log(B**2 / (S0 * K)) / sqT + lam * sqT
        y2 = y1 - sqT

        P_do = (vanilla_put()
                + S0 * norm.cdf(-x_h1) - K * disc * norm.cdf(-x_h2)
                - coef1 * (norm.cdf(y1) - norm.cdf(x1))
                + coef2 * (norm.cdf(y2) - norm.cdf(x2)))
        return P_do
    else:
        raise ValueError("option_type must be 'call' or 'put'")

In [ ]:
# closed-form pricing result
print(BS_vanilla(BS_eta))
print(BS_barrier(BS_eta))

0.12994490779249862
0.12349327004706231


In [ ]:
# 2. Monte Carlo
# 1) vanilla
# 1-1) BS
def generate_BS_paths_gpu(eta, n_paths=1000, dt=0.001):
    S0, K, r, sigma, T = eta
    n_steps = int(T / dt)

    W = cp.random.randn(n_paths, n_steps)
    S = cp.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0

    for i in range(n_steps):
        S[:, i+1] = S[:, i] * cp.exp((r - 0.5 * sigma**2) * dt + sigma * cp.sqrt(dt) * W[:, i])

    ST = S[:, -1]
    MT = S.min(axis=1)
    return ST, MT

def MC_BS_vanilla_gpu(eta, n_paths=1000, dt=0.001, type='call'):
    S0, K, r, sigma, T = eta
    ST, MT = generate_BS_paths_gpu(eta, n_paths, dt)

    if type == 'call':
        payoff = cp.maximum(ST - K, 0)
    elif type == 'put':
        payoff = cp.maximum(K - ST, 0)
    else:
        raise ValueError("type must be 'call' or 'put'")

    return float(cp.exp(-r * T) * payoff.mean())



# 1-2) Heston
def generate_heston_paths_gpu(eta, n_paths=1000, dt=0.001):
    r, lamb, v_bar, epsilon, rho, Y0, T = eta
    n_steps = int(T / dt)

    W1 = cp.random.randn(n_paths, n_steps)
    W2 = cp.random.randn(n_paths, n_steps)
    dWx = cp.sqrt(dt) * W1
    dWy = cp.sqrt(dt) * (rho * W1 + cp.sqrt(1 - rho**2) * W2)

    X = cp.zeros((n_paths, n_steps + 1))
    Y = cp.zeros((n_paths, n_steps + 1))
    X[:, 0] = 0.0
    Y[:, 0] = Y0

    for i in range(n_steps):
        Y_t = cp.maximum(Y[:, i], 0)
        X[:, i+1] = X[:, i] + (r - 0.5 * Y_t) * dt + cp.sqrt(Y_t) * dWx[:, i]
        Y[:, i+1] = (Y_t
                     + lamb * (v_bar - Y_t) * dt
                     + epsilon * cp.sqrt(Y_t) * dWy[:, i]
                     + 0.25 * epsilon**2 * (dWy[:, i]**2 - dt))

    XT = X[:, -1]
    YT = Y[:, -1]
    MT = X.min(axis=1)
    return XT, YT, MT

def MC_heston_vanilla_gpu(eta, n_paths=1000, dt=0.001, type='call'):
    r, lamb, v_bar, epsilon, rho, Y0, T = eta
    S0, K = 1.0, 1.0
    XT, YT, MT = generate_heston_paths_gpu(eta, n_paths, dt)

    ST = S0 * cp.exp(XT)

    if type == 'call':
        payoff = cp.maximum(ST - K, 0)
    elif type == 'put':
        payoff = cp.maximum(K - ST, 0)

    return float(cp.exp(-r * T) * payoff.mean())


def MC_heston_vanilla_cpu(eta, n_paths=1000, dt=0.001, type='call'):
    S0, K = 1.0, 1.0
    XT, YT, MT, masks = generate_heston_paths(eta, n_paths, dt)

    ST = S0 * np.exp(XT)

    if type == 'call':
        payoff = np.maximum(ST - K, 0)
    elif type == 'put':
        payoff = np.maximum(K - ST, 0)

    return float(np.exp(-r * T) * payoff.mean())

In [ ]:
"""
start = time.time()
print(MC_BS_vanilla_gpu(BS_eta, n_paths=1000, type='call'))
print(f"CPU: {time.time() - start:.3f}s")
"""
# CPU
# vanilla(1k, 10k, 100k) = (0.042, 0.453, 4.783)
# GPU
# vanilla(1k, 10k, 100k) = (0.143, 0.156, 0.163)

start = time.time()
print(MC_heston_vanilla_cpu(Hes_eta, n_paths=100000, type='call'))
print(f"CPU: {time.time() - start:.3f}s")
# CPU
# vanilla(1k, 10k, 100k) = (0.112, 1.160, 11.992)
# GPU
# vanilla(1k, 10k, 100k) = (0.438, 0.485, 0.622)

0.12320590946915536
CPU: 11.992s


In [ ]:
# 2) barrier 
# 2-1) BS
def MC_BS_barrier_gpu(eta, B, n_paths=2**14, dt=0.001, type='call'):
    S0, K, r, sigma, T = eta
    ST, MT = generate_BS_paths_gpu(eta, n_paths, dt)

    if type == 'call':
        payoff = cp.where(MT > B, cp.maximum(ST - K, 0), 0)
    elif type == 'put':
        payoff = cp.where(MT > B, cp.maximum(K - ST, 0), 0)

    price = float(cp.exp(-r * T) * payoff.mean())
    return price

# 2-2) Heston
def MC_heston_barrier_gpu(eta, B, n_paths=2**14, dt=0.001, type='call'):
    r, lamb, v_bar, epsilon, rho, Y0, T = eta
    S0, K = 1.0, 1.0
    XT, YT, MT = generate_heston_paths_gpu(eta, n_paths, dt)

    ST   = S0 * cp.exp(XT)
    MT_S = S0 * cp.exp(MT)

    if type == 'call':
        payoff = cp.where(MT_S > B, cp.maximum(ST - K, 0), 0)
    elif type == 'put':
        payoff = cp.where(MT_S > B, cp.maximum(K - ST, 0), 0)
    else:
        raise ValueError("type must be 'call' or 'put'")

    return float(cp.exp(-r * T) * payoff.mean())

In [ ]:
B = 0.8
"""
start = time.time()
print(MC_BS_barrier_gpu(BS_eta, B, n_paths=100000, type='call'))
print(f"Time: {time.time() - start:.3f}s")
"""
# CPU
# vanilla(1k, 10k, 100k) = ()
# GPU
# vanilla(1k, 10k, 100k) = (0.146, 0.152, 0.170)

start = time.time()
print(MC_heston_barrier_gpu(Hes_eta, B, n_paths=100000, type='call'))
print(f"Time: {time.time() - start:.3f}s")
# CPU
# vanilla(1k, 10k, 100k) = ()
# GPU
# vanilla(1k, 10k, 100k) = (0.425, 0.430, 0.489)


0.1173571203109427
Time: 0.489s


In [ ]:
# 3. FDM
# 1) vanilla
# 1-1) BS
# FTCS(만기->현재)
def FDM_BS_vanilla(eta, type='call', S_max=5, dS=0.01, dt=0.001):
    S0, K, r, sigma, T = eta

    # grid
    S_grid = np.arange(0, S_max + dS, dS)  # 주가 그리드
    t_grid = np.arange(0, T + dt, dt)       # 시간 그리드
    N = len(S_grid)
    M = len(t_grid)

    # init condition (maturity payoff)
    if type == 'call':
        V = np.maximum(S_grid - K, 0)
    elif type == 'put':
        V = np.maximum(K - S_grid, 0)
    else:
        raise ValueError("type must be 'call' or 'put'")

    # coefficient
    alpha = 0.5 * sigma**2 * S_grid**2 / dS**2 # 안정화 조건
    beta  = r * S_grid / (2 * dS)

    a = dt * (alpha - beta)         # 하삼각
    b = 1 - dt * (2 * alpha + r)    # 대각
    c = dt * (alpha + beta)         # 상삼각

    # Explicit FDM (만기 → 현재로 역행)
    for _ in range(M - 1):
        V_new = V.copy()
        V_new[1:-1] = (a[1:-1] * V[:-2]
                     + b[1:-1] * V[1:-1]
                     + c[1:-1] * V[2:])

        # 경계 조건
        if type == 'call':
            V_new[0]  = 0
            V_new[-1] = S_max - K * np.exp(-r * T)
        elif type == 'put':
            V_new[0]  = K * np.exp(-r * T)
            V_new[-1] = 0

        V = V_new

    # S0에 해당하는 인덱스 보간
    idx = S0 / dS
    i   = int(idx)
    w   = idx - i  # 보간 가중치
    price = (1 - w) * V[i] + w * V[i + 1]
    return price



In [ ]:
# Crank-Nicolson
def _CN_matrix(sigma, r, S_grid, dt, dS):
    N = len(S_grid)
    alpha = 0.25 * sigma**2 * S_grid**2 / dS**2
    beta  = 0.25 * r * S_grid / dS

    # 계수 (CN = 0.5*explicit + 0.5*implicit)
    a = -(alpha - beta)        # 하삼각
    b =  (2/dt + 2*alpha + r)  # 대각 (implicit 쪽)
    c = -(alpha + beta)        # 상삼각

    # 우변 계수 (explicit 쪽)
    a_e =  (alpha - beta)
    b_e =  (2/dt - 2*alpha - r)
    c_e =  (alpha + beta)

    return a, b, c, a_e, b_e, c_e


def _solve_tridiagonal(a, b, c, rhs, N):
    # scipy solve_banded 형식: (2, 1, 0) 행
    ab = np.zeros((3, N))
    ab[0, 1:] = c[:-1]   # 상삼각
    ab[1, :]  = b         # 대각
    ab[2, :-1] = a[1:]   # 하삼각
    return solve_banded((1, 1), ab, rhs)

def CN_BS_vanilla(eta, type='call', S_max_mult=3, dS=0.01, dt=0.01):
    S0, K, r, sigma, T = eta

    S_max  = S_max_mult * K
    S_grid = np.arange(0, S_max + dS, dS)
    N = len(S_grid)
    M = int(T / dt)

    # 만기 페이오프
    if type == 'call':
        V = np.maximum(S_grid - K, 0).copy()
    elif type == 'put':
        V = np.maximum(K - S_grid, 0).copy()
    else:
        raise ValueError("type must be 'call' or 'put'")

    a, b, c, a_e, b_e, c_e = _CN_matrix(sigma, r, S_grid, dt, dS)

    for t_idx in range(M):
        tau = (t_idx + 1) * dt  # 현재 시점의 잔존만기

        # 우변 계산
        rhs = np.zeros(N)
        rhs[1:-1] = (a_e[1:-1] * V[:-2]
                   + b_e[1:-1] * V[1:-1]
                   + c_e[1:-1] * V[2:])

        # 경계 조건
        if type == 'call':
            bc_low  = 0.0
            bc_high = S_max - K * np.exp(-r * tau)
        elif type == 'put':
            bc_low  = K * np.exp(-r * tau)
            bc_high = 0.0

        rhs[0]  = bc_low
        rhs[-1] = bc_high

        # 삼대각 행렬 설정 (경계 제외 내부만)
        a_in = a[1:-1].copy()
        b_in = b[1:-1].copy()
        c_in = c[1:-1].copy()
        rhs_in = rhs[1:-1].copy()

        # 경계 영향 반영
        rhs_in[0]  -= a_in[0]  * bc_low
        rhs_in[-1] -= c_in[-1] * bc_high

        V_in = _solve_tridiagonal(a_in, b_in, c_in, rhs_in, N - 2)

        V[0]    = bc_low
        V[1:-1] = V_in
        V[-1]   = bc_high

    # S0 보간
    idx = S0 / dS
    i   = int(idx)
    w   = idx - i
    return (1 - w) * V[i] + w * V[i + 1]

In [ ]:
# 1-2) Heston

In [ ]:
BS_eta

for opt_type in ['call', 'put']:
    cf  = BS_vanilla(BS_eta, type=opt_type)
    fdm = FDM_BS_vanilla(BS_eta, type=opt_type)
    mc  = MC_BS_vanilla_gpu(BS_eta, type=opt_type)
    print(f"[{opt_type}] closed-form: {cf:.6f} | FDM: {fdm:.6f} | MC: {mc:.6f}")

for opt_type in ['call', 'put']:
    fdm = FDM_heston_vanilla(Hes_eta, type=opt_type)
    mc  = MC_heston_vanilla_gpu(Hes_eta, n_paths=2**16, type=opt_type)
    print(f"[{opt_type}] FDM: {fdm:.6f} | MC: {mc:.6f}")

In [ ]:
# 2) barrier
# 2-1) BS
# FTCS
def FDM_BS_barrier(eta, type='call', S_max_mult=3, dS=0.01, dt=0.001, B=0.8):
    S0, K, r, sigma, T = eta

    # 그리드 설정 (S=B 이하는 knock-out이므로 B부터 시작)
    S_max = S_max_mult * K
    S_grid = np.arange(B, S_max + dS, dS)
    M = int(T / dt)
    N = len(S_grid)

    # 만기 페이오프
    if type == 'call':
        V = np.maximum(S_grid - K, 0)
    elif type == 'put':
        V = np.maximum(K - S_grid, 0)
    else:
        raise ValueError("type must be 'call' or 'put'")

    # 계수
    alpha = 0.5 * sigma**2 * S_grid**2 / dS**2
    beta  = r * S_grid / (2 * dS)

    a = dt * (alpha - beta)
    b = 1 - dt * (2 * alpha + r)
    c = dt * (alpha + beta)

    # Explicit FDM (만기 → 현재)
    for _ in range(M):
        V_new = V.copy()
        V_new[1:-1] = (a[1:-1] * V[:-2]
                     + b[1:-1] * V[1:-1]
                     + c[1:-1] * V[2:])

        # 경계 조건
        V_new[0]  = 0   # S=B : knock-out → 가치 0
        if type == 'call':
            V_new[-1] = S_max - K * np.exp(-r * T)
        elif type == 'put':
            V_new[-1] = 0

        V = V_new

    # S0 보간
    idx = (S0 - B) / dS
    i   = int(idx)
    w   = idx - i
    price = (1 - w) * V[i] + w * V[i + 1]
    return price

In [ ]:
def CN_BS_barrier(eta, type='call', S_max_mult=3, dS=0.01, dt=0.01, B=0.8):
    S0, K, r, sigma, T = eta

    S_max  = S_max_mult * K
    S_grid = np.arange(B, S_max + dS, dS)  # B부터 시작
    N = len(S_grid)
    M = int(T / dt)

    # 만기 페이오프
    if type == 'call':
        V = np.maximum(S_grid - K, 0).copy()
    elif type == 'put':
        V = np.maximum(K - S_grid, 0).copy()
    else:
        raise ValueError("type must be 'call' or 'put'")

    a, b, c, a_e, b_e, c_e = _CN_matrix(sigma, r, S_grid, dt, dS)

    for t_idx in range(M):
        tau = (t_idx + 1) * dt

        rhs = np.zeros(N)
        rhs[1:-1] = (a_e[1:-1] * V[:-2]
                   + b_e[1:-1] * V[1:-1]
                   + c_e[1:-1] * V[2:])

        # 경계 조건
        bc_low  = 0.0  # S=B: knock-out
        if type == 'call':
            bc_high = S_max - K * np.exp(-r * tau)
        elif type == 'put':
            bc_high = 0.0

        rhs[0]  = bc_low
        rhs[-1] = bc_high

        a_in = a[1:-1].copy()
        b_in = b[1:-1].copy()
        c_in = c[1:-1].copy()
        rhs_in = rhs[1:-1].copy()

        rhs_in[0]  -= a_in[0]  * bc_low
        rhs_in[-1] -= c_in[-1] * bc_high

        V_in = _solve_tridiagonal(a_in, b_in, c_in, rhs_in, N - 2)

        V[0]    = bc_low
        V[1:-1] = V_in
        V[-1]   = bc_high

    # S0 보간
    idx = (S0 - B) / dS
    i   = int(idx)
    w   = idx - i
    return (1 - w) * V[i] + w * V[i + 1]

In [ ]:
# 2-2) Heston

In [ ]:
eta_BS = np.array([1.0, 1.0, 0.03, np.sqrt(0.05), 1.5])

for opt_type in ['call', 'put']:
    cf  = BS_barrier(eta_BS, type=opt_type)
    fdm = FDM_BS_barrier(eta_BS, type=opt_type)
    mc  = MC_BS_barrier(eta_BS, type=opt_type)
    print(f"[{opt_type}] closed-form: {cf:.6f} | FDM: {fdm:.6f} | MC: {mc:.6f}")